In [ ]:
# NOTE: Do NOT hardcode API keys in notebooks or checked-in files.
# Set the GROQ API key in the environment before running the backend, for example:
# export GROQ_API_KEY=your_real_key_here
import os
print('GROQ_API_KEY present:', bool(os.getenv('GROQ_API_KEY')))

In [2]:
# Force-install a version that satisfies LangChain
!pip install requests==2.32.5 --quiet
!pip install -qU langchain langchain-openai groq youtube-transcript-api langchain-community langchain-groq faiss-cpu

In [9]:
!pip install -qU langchain langchain-text-splitters youtube-transcript-api==0.6.2

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.embeddings import DeterministicFakeEmbedding
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## **Step-1a: Indexing (Document Ingestion)**

In [19]:

# Ensure this is a valid 11-character ID
video_id = "MGXwxwYMgfU"

try:
    # 1. Initialize the API
    ytt_api = YouTubeTranscriptApi()

    # 2. Fetch and convert to a list of dictionaries
    # .to_raw_data() handles the conversion automatically
    transcript_as_dicts = ytt_api.fetch(video_id, languages=['en']).to_raw_data()

    # 3. View the result
    print(f"Successfully retrieved {len(transcript_as_dicts)} timestamped chunks.")

    # Print the first 3 chunks to see the structure
    print(transcript_as_dicts)

except Exception as e:
    print(f"An error occurred: {e}")

Successfully retrieved 129 timestamped chunks.
[{'text': "All right, I'm about to ruin your entire world\xa0\nview in the best possible way. You think life\xa0\xa0", 'start': 0.16, 'duration': 6.48}, {'text': "is complicated? Stock markets, relationships,\xa0\nsuccess, happiness. It's all impossibly complex,\xa0\xa0", 'start': 6.64, 'duration': 6.56}, {'text': 'right? Wrong. Life is laughably simple. We\xa0\njust hate simple answers. Think about it.\xa0\xa0', 'start': 13.2, 'duration': 7.44}, {'text': 'Every self-help book is 300 pages saying, "Eat\xa0\nless, move more." Every relationship advice\xa0\xa0', 'start': 20.64, 'duration': 6.24}, {'text': 'boils down to communicate and care. every success\xa0\nstory. Work consistently on something valuable.\xa0\xa0', 'start': 26.88, 'duration': 6.64}, {'text': "But that's boring. So, we create complexity. We\xa0\nneed the 47step morning routine. The revolutionary\xa0\xa0", 'start': 33.52, 'duration': 7.76}, {'text': "diet with a trademark. T

In [36]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Join all the transcript chunks into one single string
full_text = " ".join([item['text'] for item in transcript_as_dicts])

# Use split_text instead of split_documents for raw strings
chunks = splitter.split_text(full_text)

print(len(chunks))

15


In [28]:
chunks[10]

'make you feel special.'

In [47]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize a free embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Use from_texts because 'chunks' is a list of strings, not Document objects
vector_store = FAISS.from_texts(chunks, embeddings)

print("Vector store created successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created successfully!


In [39]:
vector_store.index_to_docstore_id

{0: '4a01f8af-356b-474e-9ae9-0514cc70d2d0',
 1: '19605082-ff49-47b2-88b6-866179238640',
 2: 'f156a7ba-a1aa-4841-8d99-26b48578fe6e',
 3: '8ceef54a-f0d4-42eb-ae98-03fab32a4c0e',
 4: 'e57cfbfc-8c0a-4336-9d12-2dadcd8f72ba',
 5: 'b28c9726-92ec-4907-bb79-1680b339932b',
 6: '23b61519-e561-49d1-b438-1ba46ed588f0',
 7: '2dbad19c-cc62-4181-9646-dbd0c8d9764d',
 8: '3e1214ad-d43e-4876-ba47-6221c71e1a1c',
 9: '244e43d1-fd28-45f6-82d6-a6910e1d4d02',
 10: '34c04884-4fb9-41fa-9043-e2b6abd0318e',
 11: '4f458d9e-0151-452a-97c3-492cd2d36342',
 12: 'efbdc4ec-ce18-4089-825e-ceda925b8290',
 13: '9578054b-31a9-4a94-a9af-b60d44a979f6',
 14: 'fb00a2bf-7b27-4a5b-a866-56f98fe64238'}

## **RETRIEVAL**

In [58]:
retriever=vector_store.as_retriever(search_type='similarity', serach_kwargs={"k": 4})

In [51]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x791c968e1820>, search_kwargs={})

In [52]:
retriever.invoke("what is meditation")

[Document(id='c7e02885-b7f3-47b5-8bc4-efe9088af77c', metadata={}, page_content="mind isn't driving. It's just narrating the trip.\xa0\xa0 Think you chose your career logically? Nope. You\xa0\nfelt drawn to it, then justified it with logic.\xa0\xa0 That gut feeling about someone, that's your\xa0\nemotional brain processing millions of micro\xa0\xa0 signals faster than consciousness. Your\xa0\nemotions are smarter than your thoughts.\xa0\xa0 The science is fascinating. Your emotional\xa0\nbrain processes information 500,000 times\xa0\xa0 faster than your logical brain. By the time\xa0\nyou think about something, your emotions have\xa0\xa0 already decided. You're not making decisions.\xa0\nYou're rationalizing what you've already felt.\xa0\xa0 This isn't bad news. It's great news. Your\xa0\nemotions are data, not directives. That pause\xa0\xa0 between feeling and action. That's where your\xa0\nactual power lives. Use it. Chapter 7. Memory.\xa0\xa0 Your personal fiction writer. Your memori

### **AUGMENTATION**

In [68]:
from langchain_groq import ChatGroq

# Switched to a current supported model: llama-3.1-8b-instant
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.4)

In [55]:
prompt= PromptTemplate(
    template= """
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you dont know.

    {context}
    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [83]:
question= "is the topic of life discussed in this video? if yes, then what was discussed"
retrieved_docs= retriever.invoke(question)

In [84]:
retrieved_docs

[Document(id='1ffcac74-3e61-4151-b2d4-aad197f4b833', metadata={}, page_content="The paradoxes, they teach you balance. Your biased\xa0\xa0 brain. It shows you humility. The compound effect,\xa0\nit gives you power. Every concept connects. The\xa0\xa0 ultimate cheat code has three parts. First, accept\xa0\nwhat you can't control, almost everything. Second,\xa0\xa0 master what you can control, your actions and\xa0\nreactions. Third, compound tiny improvements\xa0\xa0 daily. That's it. Everything else is commentary.\xa0\nYou now understand how your brain works,\xa0\xa0 why happiness hides, how habits form, and where\xa0\npower lives. You know memory lies, emotions drive,\xa0\xa0 and small actions compound into destiny. You\xa0\nhave the manual. And hey, if you like this video,\xa0\xa0 don't forget to subscribe and hit that like\xa0\nbutton. Also, let me know your thoughts on what\xa0\xa0 I just shared. Oh, and there's more. I've just\xa0\nstarted a Patreon to help support these videos\xa0

In [85]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)

In [86]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [87]:
final_prompt

StringPromptValue(text='\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context.\n    If the context is insufficient, just say you dont know.\n\n    The paradoxes, they teach you balance. Your biased\xa0\xa0 brain. It shows you humility. The compound effect,\xa0\nit gives you power. Every concept connects. The\xa0\xa0 ultimate cheat code has three parts. First, accept\xa0\nwhat you can\'t control, almost everything. Second,\xa0\xa0 master what you can control, your actions and\xa0\nreactions. Third, compound tiny improvements\xa0\xa0 daily. That\'s it. Everything else is commentary.\xa0\nYou now understand how your brain works,\xa0\xa0 why happiness hides, how habits form, and where\xa0\npower lives. You know memory lies, emotions drive,\xa0\xa0 and small actions compound into destiny. You\xa0\nhave the manual. And hey, if you like this video,\xa0\xa0 don\'t forget to subscribe and hit that like\xa0\nbutton. Also, let me know your thoughts on what\xa0\x

### **GENERATION**

In [88]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of life is discussed in this video. The following points were discussed:

1. Life is simple, but we create complexity.
2. Life is screaming simple answers, but we look for complex solutions.
3. Life was simple all along, but we just need to look at it in a different way.
4. We can rewrite our story by choosing which parts to emphasize.
5. 20% of what we do creates 80% of our results (the 80/20 life principle).
6. Life is exponential, and we need to identify our vital 20% and double down.
7. Effort without attachment and action without desperation is key to success.
8. The more we try to control everything, the less control we actually have (the control paradox).
9. We can only control two things: our actions and our reactions.


Note: After running the cell above, you may need to go to **Runtime > Restart session** for the changes to take effect.

# ***Building a Chain***. WHY? so we will call invoke a single time and the whole pipeline is called at once

In [90]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [91]:
def format_docs(retrieved_docs):
  context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [92]:
parallel_chain= RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [95]:
result = parallel_chain.invoke('What is 80-20 rule')

{'context': "The paradoxes, they teach you balance. Your biased\xa0\xa0 brain. It shows you humility. The compound effect,\xa0\nit gives you power. Every concept connects. The\xa0\xa0 ultimate cheat code has three parts. First, accept\xa0\nwhat you can't control, almost everything. Second,\xa0\xa0 master what you can control, your actions and\xa0\nreactions. Third, compound tiny improvements\xa0\xa0 daily. That's it. Everything else is commentary.\xa0\nYou now understand how your brain works,\xa0\xa0 why happiness hides, how habits form, and where\xa0\npower lives. You know memory lies, emotions drive,\xa0\xa0 and small actions compound into destiny. You\xa0\nhave the manual. And hey, if you like this video,\xa0\xa0 don't forget to subscribe and hit that like\xa0\nbutton. Also, let me know your thoughts on what\xa0\xa0 I just shared. Oh, and there's more. I've just\xa0\nstarted a Patreon to help support these videos\xa0\xa0 and connect with you more directly. Check out the\xa0\nlink in

In [96]:
parser=StrOutputParser()

In [97]:
main_chain= parallel_chain | prompt | llm | parser

In [98]:
main_chain.invoke('Can you summarise the video')

"The video discusses how life is often made more complicated than it needs to be. It highlights three key points:\n\n1. Accept what you can't control and master what you can control.\n2. Compound tiny improvements daily.\n3. Effort without attachment is what works.\n\nThe video also touches on various paradoxes, such as the effort paradox (the harder you try, the more you fail) and the backwards law (the more you want something, the more it eludes you).\n\nAdditionally, the video talks about how our brains work, including:\n\n- How memories are malleable and can be altered by recall.\n- How our brains fill gaps in memories with current knowledge and modify details.\n- How memories can be false or partially manufactured.\n\nThe video encourages viewers to simplify their approach to life and to focus on what really works, rather than getting caught up in complexity."